# Conditional VAE for Protein Sequence Modification Prediction

This notebook implements a Conditional Variational Autoencoder (C-VAE) that:
- Takes protein embeddings (1536-dim) and existing modification masks as input
- Learns to generate new modification patterns
- Uses a 3-class classification (0, 1, 2) for each position in the sequence

In [1]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Sun Feb  1 19:03:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Setup and Imports

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

import pandas as pd
import numpy as np
import ast
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.model_selection import GroupShuffleSplit
import matplotlib.pyplot as plt
import seaborn as sns

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cuda
GPU: NVIDIA A100-SXM4-80GB
Memory: 85.17 GB


### Embedding Generation (Ankh-large)

The input protein sequences are first processed using the **Ankh-large** model (a T5-based protein language model) to generate fixed-length embeddings.

```mermaid
graph LR
    Seq["Protein Sequence"] --> Tokens["Tokenizer"]
    Tokens --> T5["Ankh-large T5 Encoder"]
    T5 --> LastHidden["Last Hidden State (L, 1536)"]
    LastHidden --> MeanPool["Mean Pooling (excluding EOS)"]
    MeanPool --> Vec["Protein Vector (1536d)"]
```

## 2. Data Loading and Preparation

In [4]:
data_path = '/content/protein_data.parquet'
df = pd.read_parquet(data_path)

df = df.iloc[0:70675, :]
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

# df = pd.read_excel('/content/protein_constructs_w_label_masks.xlsx')
# embeddings = torch.load('/content/ankh_embeddings.pt', weights_only=True)

Dataset shape: (70675, 8)

Columns: ['rcsb_id', 'rcsb_entity_ids', 'uniprot_seq', 'pbd_id', 'pdb_sequence_sanitized', 'label_mask', 'seq_len', 'embeddings']

First few rows:


,rcsb_id,rcsb_entity_ids,uniprot_seq,pbd_id,pdb_sequence_sanitized,label_mask,seq_len,embeddings
0,7NDU,5,NYGYTFGSGTRLTVV,MDTGVSQNPRHKITKRGQNVTFRCDPISEHNRLYWYRQTLGQGPEF...,MDTGVSQNPRHKITKRGQNVTFRCDPISEHNRLYWYRQTLGQGPEF...,"[2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",15,"[0.018173939, 0.003579815, -0.006878902, 0.001..."
1,7NDQ,5,NYGYTFGSGTRLTVV,MDTGVSQNPRHKITKRGQNVTFRCDPISEHNRLYWYRQTLGQGPEF...,MDTGVSQNPRHKITKRGQNVTFRCDPISEHNRLYWYRQTLGQGPEF...,"[2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",15,"[0.018173939, 0.003579815, -0.006878902, 0.001..."
2,7NDT,5,STDTQYFGPGTRLTVL,MEAGVTQFPSHSVIEKGQTVTLRCDPISGHDNLYWYRRVMGKEIKF...,MEAGVTQFPSHSVIEKGQTVTLRCDPISGHDNLYWYRRVMGKEIKF...,"[2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",16,"[0.0038214922, 0.0039241225, -0.008048792, 0.0..."
3,7NDQ,4,XIYNQGGKLIFGQGTELSVKP,MLAKTTQPISMDSYEGQEVNITCSHNNIATNDYITWYQQFPSQGPR...,MLAKTTQPISMDSYEGQEVNITCSHNNIATNDYITWYQQFPSQGPR...,"[2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",21,"[0.012161151, -0.0019863984, -0.008338355, 0.0..."
4,7NDU,4,XIYNQGGKLIFGQGTELSVKP,MLAKTTQPISMDSYEGQEVNITCSHNNIATNDYITWYQQFPSQGPR...,MLAKTTQPISMDSYEGQEVNITCSHNNIATNDYITWYQQFPSQGPR...,"[2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",21,"[0.012161151, -0.0019863984, -0.008338355, 0.0..."


In [5]:
def parse_label_mask(mask_str):
    """Convert string representation of list to actual list"""
    if isinstance(mask_str, str):
        return ast.literal_eval(mask_str)
    return mask_str

# for i, row in df.iterrows():
#   try:
#     parse_label_mask(row['label_mask'])
#   except:
#     print(i, row['label_mask'])
df['label_mask'] = df['label_mask'].apply(parse_label_mask)

print(f"Sample label_mask: {df.iloc[0]['label_mask'][:20]}")
print(f"Type: {type(df.iloc[0]['label_mask'])}")
print(f"Length: {len(df.iloc[0]['label_mask'])}")

Sample label_mask: [2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Type: <class 'list'>
Length: 15


In [6]:
df.dtypes

,0
rcsb_id,object
rcsb_entity_ids,int64
uniprot_seq,object
pbd_id,object
pdb_sequence_sanitized,object
label_mask,object
seq_len,int64
embeddings,object


In [7]:
# df = df[df["uniprot_seq"].str.len() > df["pdb_sequence_sanitized"].str.len()]

# # print("Dataset Statistics:")
# # print(f"Total samples: {len(filtered_df)}")
# # print(f"Unique proteins (rcsb_id): {filtered_df['rcsb_id'].nunique()}")
# # print(f"\nSequence length statistics:")
# # print(filtered_df['seq_len'].describe())

# # all_values = []
# # for mask in filtered_df['label_mask'].head(1000):
# #     all_values.extend(mask)

# # from collections import Counter
# # value_counts = Counter(all_values)
# # print(f"\nLabel mask value distribution (sample):")
# # print(value_counts)
# # print(f"Unique values: {sorted(set(all_values))}")

In [8]:
print("Dataset Statistics:")
print(f"Total samples: {len(df)}")
print(f"Unique proteins (rcsb_id): {df['rcsb_id'].nunique()}")
print(f"\nSequence length statistics:")
print(df['seq_len'].describe())

all_values = []
for mask in df['label_mask']:#.head(1000):
    all_values.extend(mask)

from collections import Counter
value_counts = Counter(all_values)
print(f"\nLabel mask value distribution (sample):")
print(value_counts)
print(f"Unique values: {sorted(set(all_values))}")

Dataset Statistics:
Total samples: 70675
Unique proteins (rcsb_id): 56507

Sequence length statistics:
count    70675.000000
mean       591.005249
std        515.835815
min         15.000000
25%        260.000000
50%        454.000000
75%        732.000000
max       8797.000000
Name: seq_len, dtype: float64

Label mask value distribution (sample):
Counter({1: 24237344, 0: 17215008, 2: 316944})
Unique values: [0, 1, 2]


In [9]:
# sum([2 in x for x in df['label_mask']])

In [10]:
df['label_mask'] = df['label_mask'].apply(lambda x: [0 if val == 2 else val for val in x])

print("Dataset Statistics:")
print(f"Total samples: {len(df)}")
print(f"Unique proteins (rcsb_id): {df['rcsb_id'].nunique()}")
print(f"\nSequence length statistics:")
print(df['seq_len'].describe())

all_values = []
for mask in df['label_mask']:#.head(1000):
    all_values.extend(mask)

from collections import Counter
value_counts = Counter(all_values)
print(f"\nLabel mask value distribution (sample):")
print(value_counts)
print(f"Unique values: {sorted(set(all_values))}")

Dataset Statistics:
Total samples: 70675
Unique proteins (rcsb_id): 56507

Sequence length statistics:
count    70675.000000
mean       591.005249
std        515.835815
min         15.000000
25%        260.000000
50%        454.000000
75%        732.000000
max       8797.000000
Name: seq_len, dtype: float64

Label mask value distribution (sample):
Counter({1: 24237344, 0: 17531952})
Unique values: [0, 1]


In [11]:
only_zeros = df['label_mask'].apply(lambda x: set(x) == {0})

count = only_zeros.sum()
print(f"Number of rows with only 0s: {count}")

Number of rows with only 0s: 10775


## 3. Train/Val/Test Split (Group-Based)

In [12]:
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_val_idx, test_idx = next(gss_test.split(df, groups=df['rcsb_id']))

df_train_val = df.iloc[train_val_idx].reset_index(drop=True)
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.111, random_state=42)  # 0.111 * 0.9 ≈ 0.1
train_idx, val_idx = next(gss_val.split(df_train_val, groups=df_train_val['rcsb_id']))

train_df = df_train_val.iloc[train_idx].reset_index(drop=True)
val_df = df_train_val.iloc[val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

print(f"Train set: {len(train_df)} samples ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val set: {len(val_df)} samples ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test set: {len(test_df)} samples ({len(test_df)/len(df)*100:.1f}%)")
print(f"\nTotal: {len(train_df) + len(val_df) + len(test_df)} samples")

train_ids = set(train_df['rcsb_id'])
val_ids = set(val_df['rcsb_id'])
test_ids = set(test_df['rcsb_id'])

assert train_ids.isdisjoint(val_ids), "Train and val sets overlap!"
assert train_ids.isdisjoint(test_ids), "Train and test sets overlap!"
assert val_ids.isdisjoint(test_ids), "Val and test sets overlap!"
print("\n✓ No overlap between splits confirmed")

Train set: 56474 samples (79.9%)
Val set: 7058 samples (10.0%)
Test set: 7143 samples (10.1%)

Total: 70675 samples

✓ No overlap between splits confirmed


In [13]:
# output_dir = Path('/Users/haripat/Desktop/SF/protein/data/splits')
output_dir = Path('/content/splits')
output_dir.mkdir(exist_ok=True, parents=True)

train_df.to_parquet(output_dir / 'train.parquet', index=False)
val_df.to_parquet(output_dir / 'val.parquet', index=False)
test_df.to_parquet(output_dir / 'test.parquet', index=False)

print(f"Saved splits to {output_dir}")

Saved splits to /content/splits


In [14]:
# train_df['embeddings'].loc[0].shape

## 3.1 Calculate Class Weights

In [15]:
# Calculate class weights to handle imbalanced modification patterns
all_train_labels = []
for mask in train_df['label_mask']:
    all_train_labels.extend(mask)

print(Counter(all_values))

from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(all_train_labels)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=all_train_labels)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)

print(f"Target Classes: {classes}")
print(f"Calculated Class Weights: {class_weights}")

Counter({1: 24237344, 0: 17531952})
Target Classes: [0 1]
Calculated Class Weights: tensor([1.1929, 0.8608], device='cuda:0')


## 4. PyTorch Dataset and DataLoader

In [16]:
class ProteinModificationDataset(Dataset):
    """Dataset for protein embeddings and modification masks"""

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        embedding = torch.tensor(row['embeddings'], dtype=torch.float32)

        label_mask = row['label_mask']
        if isinstance(label_mask, str):
            label_mask = ast.literal_eval(label_mask)
        label_mask = torch.tensor(label_mask, dtype=torch.long)

        return {
            'embedding': embedding,
            'label_mask': label_mask,
            'seq_len': len(label_mask)
        }

train_dataset = ProteinModificationDataset(train_df)
sample = train_dataset[0]
print(f"Sample embedding shape: {sample['embedding'].shape}")
print(f"Sample label_mask shape: {sample['label_mask'].shape}")
print(f"Sample seq_len: {sample['seq_len']}")

Sample embedding shape: torch.Size([1536])
Sample label_mask shape: torch.Size([15])
Sample seq_len: 15


In [17]:
def collate_fn(batch):
    """
    Custom collate function to handle variable-length sequences.
    Pads label_masks to the max length in the batch.
    """
    embeddings = torch.stack([item['embedding'] for item in batch])

    max_len = max(item['seq_len'] for item in batch)

    batch_size = len(batch)
    padded_masks = torch.zeros(batch_size, max_len, dtype=torch.long)
    attention_mask = torch.zeros(batch_size, max_len, dtype=torch.bool)

    for i, item in enumerate(batch):
        seq_len = item['seq_len']
        padded_masks[i, :seq_len] = item['label_mask']
        attention_mask[i, :seq_len] = True

    return {
        'embeddings': embeddings,
        'label_masks': padded_masks,
        'attention_mask': attention_mask
    }

test_loader = DataLoader(train_dataset, batch_size=4, collate_fn=collate_fn)
test_batch = next(iter(test_loader))
print(f"Batch embeddings shape: {test_batch['embeddings'].shape}")
print(f"Batch label_masks shape: {test_batch['label_masks'].shape}")
print(f"Batch attention_mask shape: {test_batch['attention_mask'].shape}")

Batch embeddings shape: torch.Size([4, 1536])
Batch label_masks shape: torch.Size([4, 21])
Batch attention_mask shape: torch.Size([4, 21])


### C-VAE Architecture Overview

Below is a high-level visualization of the architecture. The **Encoder** fuses the protein embedding with the modification mask to map samples into the latent space. The **Decoder** then generates predicted modification patterns based on the sampled latent vector $z$ and the original protein embedding.

```mermaid
graph TD
    classDef input fill:#f9f,stroke:#333,stroke-width:2px,color:#000;
    classDef process fill:#bbf,stroke:#333,stroke-width:1px,color:#000;
    classDef latent fill:#bfb,stroke:#333,stroke-width:2px,color:#000;

    %% Inputs
    X["Protein Embedding (1536d)"]:::input
    Y["Label Mask (Target)"]:::input
    E["Noise (ε)"]:::input

    %% Encoder
    subgraph Encoder["Encoder (Conditioning)"]
        X --> X_MLP_E["MLP (Feature Extraction)"]:::process
        Y --> Y_EMB["Mask Embedding"]:::process
        Y_EMB --> Y_LSTM["Bidirectional LSTM"]:::process
        X_MLP_E --> CONCAT_E["Concatenation"]:::process
        Y_LSTM --> CONCAT_E
        CONCAT_E --> FUSION_E["Fusion Layer (MLP)"]:::process
        FUSION_E --> MU["μ (Latent Mean)"]:::process
        FUSION_E --> LOGVAR["log σ² (Latent Variance)"]:::process
    end

    %% Reparameterization
    MU --> REPARAM["z = μ + σ * ε"]:::latent
    LOGVAR --> REPARAM
    E --> REPARAM
    REPARAM --> Z["Latent Vector (z)"]:::latent

    %% Decoder
    subgraph Decoder["Decoder (Autoregressive Generation)"]
        X --> X_MLP_D["MLP (Feature Extraction)"]:::process
        Z --> CONCAT_D["Concatenation"]:::process
        X_MLP_D --> CONCAT_D
        CONCAT_D --> FUSION_D["Fusion Layer (Context)"]:::process
        FUSION_D --> INIT_H["Hidden/Cell States Init"]:::process
        FUSION_D --> LSTM_INPUT["Step-wise Input"]:::process
        PREV_Y["Prev Token / Teacher Forcing"]:::input --> LSTM_INPUT
        INIT_H --> LSTM["Generative LSTM"]:::process
        LSTM_INPUT --> LSTM
        LSTM --> LOGITS["3-Class Logits (0, 1, 2)"]:::process
    end
```

## 5. C-VAE Model Architecture

In [18]:
class Encoder(nn.Module):
    """Encoder: processes embeddings and masks to produce latent distribution"""

    def __init__(self, embedding_dim=1536, hidden_dim=512, lstm_hidden=256, latent_dim=128, dropout=0.2):
        super().__init__()

        self.embedding_mlp = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.mask_embedding = nn.Embedding(3, 32)  # 3 classes -> 32 dim
        self.mask_lstm = nn.LSTM(
            input_size=32,
            hidden_size=lstm_hidden,
            num_layers=2,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )

        fusion_input_dim = hidden_dim // 2 + lstm_hidden * 2  # *2 for bidirectional
        self.fusion = nn.Sequential(
            nn.Linear(fusion_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, embeddings, masks, attention_mask):
        """
        Args:
            embeddings: (batch_size, embedding_dim)
            masks: (batch_size, seq_len) - integer labels
            attention_mask: (batch_size, seq_len) - boolean mask
        """
        emb_features = self.embedding_mlp(embeddings)  # (batch_size, hidden_dim//2)

        mask_embedded = self.mask_embedding(masks)  # (batch_size, seq_len, 32)

        lengths = attention_mask.sum(dim=1).cpu()
        packed_input = nn.utils.rnn.pack_padded_sequence(
            mask_embedded, lengths, batch_first=True, enforce_sorted=False
        )

        packed_output, (hidden, cell) = self.mask_lstm(packed_input)

        hidden_fwd = hidden[-2]
        hidden_bwd = hidden[-1]
        mask_features = torch.cat([hidden_fwd, hidden_bwd], dim=1)

        combined = torch.cat([emb_features, mask_features], dim=1)
        fused = self.fusion(combined)

        mu = self.fc_mu(fused)
        logvar = self.fc_logvar(fused)

        return mu, logvar

In [19]:
class Decoder(nn.Module):
    """Decoder: generates modification mask sequence from latent vector and embedding"""

    def __init__(self, embedding_dim=1536, hidden_dim=512, lstm_hidden=256, latent_dim=128, num_classes=2, dropout=0.2):
        super().__init__()

        self.lstm_hidden = lstm_hidden
        self.num_classes = num_classes

        self.embedding_mlp = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        fusion_input_dim = hidden_dim // 2 + latent_dim
        self.fusion = nn.Sequential(
            nn.Linear(fusion_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.fc_hidden = nn.Linear(hidden_dim, lstm_hidden)
        self.fc_cell = nn.Linear(hidden_dim, lstm_hidden)

        self.input_embedding = nn.Embedding(num_classes, 32)
        self.lstm = nn.LSTM(
            input_size=32 + hidden_dim,
            hidden_size=lstm_hidden,
            num_layers=2,
            batch_first=True,
            dropout=dropout
        )

        self.output_layer = nn.Linear(lstm_hidden, num_classes)

    def forward(self, embeddings, z, target_masks, attention_mask, teacher_forcing_ratio=0.5):
        """
        Args:
            embeddings: (batch_size, embedding_dim)
            z: (batch_size, latent_dim) - sampled latent vector
            target_masks: (batch_size, seq_len) - target sequences for teacher forcing
            attention_mask: (batch_size, seq_len)
            teacher_forcing_ratio: probability of using teacher forcing
        """
        batch_size, seq_len = target_masks.shape

        emb_features = self.embedding_mlp(embeddings)

        combined = torch.cat([emb_features, z], dim=1)
        context = self.fusion(combined)

        h0 = self.fc_hidden(context).unsqueeze(0).repeat(2, 1, 1)
        c0 = self.fc_cell(context).unsqueeze(0).repeat(2, 1, 1)

        context_expanded = context.unsqueeze(1).repeat(1, seq_len, 1)

        outputs = []
        hidden = (h0, c0)

        input_token = torch.zeros(batch_size, dtype=torch.long, device=embeddings.device)

        for t in range(seq_len):
            input_embedded = self.input_embedding(input_token)

            lstm_input = torch.cat([input_embedded, context], dim=1).unsqueeze(1)

            lstm_out, hidden = self.lstm(lstm_input, hidden)

            logits = self.output_layer(lstm_out.squeeze(1))
            outputs.append(logits)

            if t < seq_len - 1:
                if np.random.random() < teacher_forcing_ratio:
                    input_token = target_masks[:, t]
                else:
                    input_token = logits.argmax(dim=1)

        outputs = torch.stack(outputs, dim=1)
        return outputs

In [20]:
class ConditionalVAE(nn.Module):
    """Complete Conditional VAE model"""

    def __init__(self, embedding_dim=1536, hidden_dim=512, lstm_hidden=256,
                 latent_dim=128, num_classes=3, dropout=0.2):
        super().__init__()

        self.encoder = Encoder(embedding_dim, hidden_dim, lstm_hidden, latent_dim, dropout)
        self.decoder = Decoder(embedding_dim, hidden_dim, lstm_hidden, latent_dim, num_classes, dropout)

    def reparameterize(self, mu, logvar):
        """Reparameterization trick: z = mu + sigma * epsilon"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, embeddings, masks, attention_mask, teacher_forcing_ratio=0.5):
        """
        Args:
            embeddings: (batch_size, embedding_dim)
            masks: (batch_size, seq_len)
            attention_mask: (batch_size, seq_len)
            teacher_forcing_ratio: probability of using teacher forcing
        """
        mu, logvar = self.encoder(embeddings, masks, attention_mask)

        z = self.reparameterize(mu, logvar)

        logits = self.decoder(embeddings, z, masks, attention_mask, teacher_forcing_ratio)

        return logits, mu, logvar

    def generate(self, embeddings, num_samples=1, max_len=500):
        """Generate new modification masks"""
        self.eval()
        with torch.no_grad():
            batch_size = embeddings.shape[0]

            z = torch.randn(batch_size, self.decoder.fc_hidden.in_features, device=embeddings.device)

            dummy_masks = torch.zeros(batch_size, max_len, dtype=torch.long, device=embeddings.device)
            dummy_attention = torch.ones(batch_size, max_len, dtype=torch.bool, device=embeddings.device)

            logits = self.decoder(embeddings, z, dummy_masks, dummy_attention, teacher_forcing_ratio=0.0)

            return logits.argmax(dim=-1)

# model = ConditionalVAE(
#     embedding_dim=1536,
#     hidden_dim=512,
#     lstm_hidden=256,
#     latent_dim=128,
#     num_classes=3,
#     dropout=0.2
# ).to(device)

model = ConditionalVAE(
    embedding_dim=1536,
    hidden_dim=128,
    lstm_hidden=64,
    latent_dim=32,
    num_classes=2,
    dropout=0.2
).to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Model parameters: 714,082
Trainable parameters: 714,082


In [21]:
test_batch = next(iter(test_loader))
test_batch = {k: v.to(device) for k, v in test_batch.items()}

logits, mu, logvar = model(
    test_batch['embeddings'],
    test_batch['label_masks'],
    test_batch['attention_mask']
)

print(f"Output logits shape: {logits.shape}")
print(f"Mu shape: {mu.shape}")
print(f"Logvar shape: {logvar.shape}")

Output logits shape: torch.Size([4, 21, 2])
Mu shape: torch.Size([4, 32])
Logvar shape: torch.Size([4, 32])


## 6. Loss Function

In [22]:
def vae_loss(logits, targets, mu, logvar, attention_mask, kl_weight=1.0, class_weights=None):
    """
    VAE loss = Reconstruction Loss + KL Divergence Loss
    """
    logits_flat = logits.reshape(-1, logits.size(-1))
    targets_flat = targets.reshape(-1)

    # Reconstruction loss with optional class weights and numerical stability
    recon_loss = F.cross_entropy(
        logits_flat,
        targets_flat,
        reduction='none',
        weight=class_weights
    )
    recon_loss = recon_loss.reshape(targets.shape)

    # Ignore padding
    recon_loss = (recon_loss * attention_mask.float()).sum() / (attention_mask.sum() + 1e-8)

    # KL divergence loss with clamping to prevent NaN/explosion
    logvar = torch.clamp(logvar, min=-10.0, max=10.0)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    kl_loss = kl_loss / targets.size(0)

    total_loss = recon_loss + kl_weight * kl_loss

    return total_loss, recon_loss, kl_loss

total_loss, recon_loss, kl_loss = vae_loss(
    logits, test_batch['label_masks'], mu, logvar, test_batch['attention_mask'], class_weights=class_weights
)
print(f"Total loss: {total_loss.item():.4f}")
print(f"Reconstruction loss: {recon_loss.item():.4f}")
print(f"KL loss: {kl_loss.item():.4f}")

Total loss: 6.1602
Reconstruction loss: 0.7658
KL loss: 5.3944


In [23]:
train_dataset[1]

{'embedding': tensor([ 0.0182,  0.0036, -0.0069,  ..., -0.0138, -0.0081,  0.0042]),
 'label_mask': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 'seq_len': 15}

In [24]:
print(f"Embeddings have NaN: {torch.isnan(train_dataset[0]['embedding']).any()}")
print(f"Masks have invalid values: {(train_dataset[0]['label_mask'] >= 3).any()}")

Embeddings have NaN: False
Masks have invalid values: False


## 7. Training Setup

In [25]:
BATCH_SIZE = 256  # Increased for A100 80GB
LEARNING_RATE = 1e-4
NUM_EPOCHS = 30
KL_WEIGHT_START = 0.0
KL_WEIGHT_END = 1.0
KL_ANNEAL_EPOCHS = 10

train_dataset = ProteinModificationDataset(train_df)
val_dataset = ProteinModificationDataset(val_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Train batches: 221
Val batches: 28


In [26]:
# model = ConditionalVAE(
#     embedding_dim=1536,
#     hidden_dim=512,
#     lstm_hidden=256,
#     latent_dim=128,
#     num_classes=3,
#     dropout=0.2
# ).to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)#, verbose=True)

print("Model initialized and ready for training!")

Model initialized and ready for training!


In [27]:
# import torch
# import gc

# # 1. Delete the model and optimizer objects
# del model
# del optimizer
# del scheduler

# # 2. If you have any remaining tensors from the training loop (like loss or outputs)
# # del loss, outputs, inputs

# # 3. Force Python's garbage collection
# gc.collect()

# # 4. Clear the GPU memory cache
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()

# print("Model removed and GPU memory cleared!")

## 8. Training Loop

In [28]:
def train_epoch(model, train_loader, optimizer, epoch, kl_weight, teacher_forcing_ratio, class_weights):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]")
    for batch in pbar:
        embeddings = batch['embeddings'].to(device)
        masks = batch['label_masks'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        optimizer.zero_grad()

        # Use current scheduled teacher forcing ratio
        logits, mu, logvar = model(embeddings, masks, attention_mask, teacher_forcing_ratio=teacher_forcing_ratio)

        loss, recon_loss, kl_loss = vae_loss(logits, masks, mu, logvar, attention_mask, kl_weight, class_weights)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()

        predictions = logits.argmax(dim=-1)
        correct += ((predictions == masks) & attention_mask).sum().item()
        total += attention_mask.sum().item()

        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100*correct/total:.2f}%',
            'kl_w': f'{kl_weight:.3f}',
            'tf_r': f'{teacher_forcing_ratio:.2f}'
        })

    return {
        'loss': total_loss / len(train_loader),
        'recon_loss': total_recon / len(train_loader),
        'kl_loss': total_kl / len(train_loader),
        'accuracy': 100 * correct / total
    }

def validate(model, val_loader, kl_weight, class_weights):
    """Validate the model (strictly no teacher forcing)"""
    model.eval()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    correct = 0
    total = 0

    with torch.no_grad():
        pbar = tqdm(val_loader, desc="Validation")
        for batch in pbar:
            embeddings = batch['embeddings'].to(device)
            masks = batch['label_masks'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            logits, mu, logvar = model(embeddings, masks, attention_mask, teacher_forcing_ratio=0.0)

            loss, recon_loss, kl_loss = vae_loss(logits, masks, mu, logvar, attention_mask, kl_weight, class_weights)

            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl_loss.item()

            predictions = logits.argmax(dim=-1)
            correct += ((predictions == masks) & attention_mask).sum().item()
            total += attention_mask.sum().item()

            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100*correct/total:.2f}%'
            })

    return {
        'loss': total_loss / len(val_loader),
        'recon_loss': total_recon / len(val_loader),
        'kl_loss': total_kl / len(val_loader),
        'accuracy': 100 * correct / total
    }

In [29]:
# OPTIONAL: Resume from checkpoint
# Run this cell ONLY if you want to load a previously saved model
RESUME_TRAINING = False # Set to True to load checkpoint

if RESUME_TRAINING:
    checkpoint_path = Path('/content/cvae/best_model.pt')
    if checkpoint_path.exists():
        print(f"Loading checkpoint from {checkpoint_path}...")
        checkpoint = torch.load(checkpoint_path, map_location=device)

        # Handle potential torch.compile wrapper prefix
        state_dict = checkpoint['model_state_dict']
        if any(k.startswith('_orig_mod.') for k in state_dict.keys()):
            state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}

        # Load weights into the model
        # Note: If you compiled the model, you'll need to load into the compiled model
        # or remove prefixes as done above.
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

        # Restore stats
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['val_loss']
        print(f"Resuming from Epoch {start_epoch} (Previous Best Loss: {best_val_loss:.4f})")
    else:
        print("No checkpoint found. Starting from scratch.")
        start_epoch = 0
else:
    print("Starting from scratch (RESUME_TRAINING = False).")
    start_epoch = 0

Starting from scratch (RESUME_TRAINING = False).


In [ ]:
history = {
    'train_loss': [],
    'train_recon': [],
    'train_kl': [],
    'train_acc': [],
    'val_loss': [],
    'val_recon': [],
    'val_kl': [],
    'val_acc': [],
    'kl_weight': [],
    'tf_ratio': []
}

best_val_loss = float('inf')
checkpoint_dir = Path('/content/cvae')
checkpoint_dir.mkdir(exist_ok=True, parents=True)

# Optimization Schedules
TF_START, TF_END = 1.0, 0.1
KL_WARMUP = 5  # Epochs with 0 KL weight

for epoch in range(start_epoch, NUM_EPOCHS):
    # 1. KL Annealing Schedule (Linear with warmup)
    if epoch < KL_WARMUP:
        kl_weight = 0.0
    elif epoch < KL_ANNEAL_EPOCHS + KL_WARMUP:
        kl_weight = (epoch - KL_WARMUP) / KL_ANNEAL_EPOCHS
    else:
        kl_weight = KL_WEIGHT_END

    # 2. Scheduled Sampling (Linear decay)
    tf_ratio = max(TF_END, TF_START - (TF_START - TF_END) * (epoch / (NUM_EPOCHS * 0.75)))

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} | KL Weight: {kl_weight:.4f} | TF Ratio: {tf_ratio:.2f}")

    train_metrics = train_epoch(model, train_loader, optimizer, epoch, kl_weight, tf_ratio, class_weights)
    val_metrics = validate(model, val_loader, kl_weight, class_weights)

    scheduler.step(val_metrics['loss'])

    history['train_loss'].append(train_metrics['loss'])
    history['train_recon'].append(train_metrics['recon_loss'])
    history['train_kl'].append(train_metrics['kl_loss'])
    history['train_acc'].append(train_metrics['accuracy'])
    history['val_loss'].append(val_metrics['loss'])
    history['val_recon'].append(val_metrics['recon_loss'])
    history['val_kl'].append(val_metrics['kl_loss'])
    history['val_acc'].append(val_metrics['accuracy'])
    history['kl_weight'].append(kl_weight)
    history['tf_ratio'].append(tf_ratio)

    print(f"Train Loss: {train_metrics['loss']:.4f} | Train Acc: {train_metrics['accuracy']:.2f}%")
    print(f"Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.2f}%")

    if val_metrics['loss'] < best_val_loss:
        best_val_loss = val_metrics['loss']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_metrics['loss'],
            'val_acc': val_metrics['accuracy'],
            'class_weights': class_weights
        }, checkpoint_dir / 'best_model.pt')
        print(f"✓ Saved best model (val_loss: {best_val_loss:.4f})")

    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history': history
        }, checkpoint_dir / f'checkpoint_epoch_{epoch+1}.pt')

print("\n" + "="*50)
print("Training completed!")
print(f"Best validation loss: {best_val_loss:.4f}")
print("="*50)



Epoch 1/30 | KL Weight: 0.0000 | TF Ratio: 1.00


Epoch 1 [Train]:   0%|          | 0/221 [00:00<?, ?it/s]

In [ ]:

# Epoch 1/30 | KL Weight: 0.0000 | TF Ratio: 1.00
# Epoch 1 [Train]: 100%
#  1455/1455 [1:36:42<00:00,  3.70s/it, loss=0.3118, acc=95.51%, kl_w=0.000, tf_r=1.00]
# Validation: 100%
#  183/183 [00:52<00:00,  1.71s/it, loss=3.3973, acc=26.63%]
# Train Loss: 0.3713 | Train Acc: 95.51%
# Val Loss: 3.3227 | Val Acc: 26.63%
# ✓ Saved best model (val_loss: 3.3227)

# Epoch 2/30 | KL Weight: 0.0000 | TF Ratio: 0.96
# Epoch 2 [Train]: 100%
#  1455/1455 [1:37:37<00:00,  3.22s/it, loss=0.2809, acc=96.82%, kl_w=0.000, tf_r=0.96]
# Validation: 100%
#  183/183 [00:51<00:00,  1.69s/it, loss=1.3177, acc=51.39%]
# Train Loss: 0.2488 | Train Acc: 96.82%
# Val Loss: 3.2637 | Val Acc: 51.39%
# ✓ Saved best model (val_loss: 3.2637)

# Epoch 3/30 | KL Weight: 0.0000 | TF Ratio: 0.92
# Epoch 3 [Train]: 100%
#  1455/1455 [1:38:09<00:00,  3.26s/it, loss=0.1314, acc=96.20%, kl_w=0.000, tf_r=0.92]
# Validation: 100%
#  183/183 [00:52<00:00,  1.71s/it, loss=1.1531, acc=49.51%]
# Train Loss: 0.2207 | Train Acc: 96.20%
# Val Loss: 3.1824 | Val Acc: 49.51%
# ✓ Saved best model (val_loss: 3.1824)

# Epoch 4/30 | KL Weight: 0.0000 | TF Ratio: 0.88
# Epoch 4 [Train]:   0%
#  3/1455 [00:16<1:55:57,  4.79s/it, loss=0.3893, acc=94.38%, kl_w=0.000, tf_r=0.88]

## 9. Visualization and Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].plot(history['train_loss'], label='Train')
axes[0, 0].plot(history['val_loss'], label='Val')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Total Loss')
axes[0, 0].set_title('Total Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(history['train_recon'], label='Train')
axes[0, 1].plot(history['val_recon'], label='Val')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Reconstruction Loss')
axes[0, 1].set_title('Reconstruction Loss')
axes[0, 1].legend()
axes[0, 1].grid(True)

axes[1, 0].plot(history['train_kl'], label='Train')
axes[1, 0].plot(history['val_kl'], label='Val')
axes[1, 0].plot(history['kl_weight'], label='KL Weight', linestyle='--')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('KL Divergence')
axes[1, 0].set_title('KL Divergence Loss')
axes[1, 0].legend()
axes[1, 0].grid(True)

axes[1, 1].plot(history['train_acc'], label='Train')
axes[1, 1].plot(history['val_acc'], label='Val')
axes[1, 1].axhline(y=33.33, color='r', linestyle='--', label='Random Baseline')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy (%)')
axes[1, 1].set_title('Accuracy')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig(checkpoint_dir / 'training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Training curves saved to {checkpoint_dir / 'training_curves.png'}")

In [ ]:
checkpoint = torch.load(checkpoint_dir / 'best_model.pt')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1}")
print(f"Validation loss: {checkpoint['val_loss']:.4f}")
print(f"Validation accuracy: {checkpoint['val_acc']:.2f}%")

In [ ]:
model.eval()
sample_batch = next(iter(val_loader))
sample_batch = {k: v.to(device) for k, v in sample_batch.items()}

with torch.no_grad():
    logits, mu, logvar = model(
        sample_batch['embeddings'],
        sample_batch['label_masks'],
        sample_batch['attention_mask'],
        teacher_forcing_ratio=0.0
    )
    predictions = logits.argmax(dim=-1)

print("Sample Predictions vs Ground Truth:\n")
for i in range(min(3, len(predictions))):
    seq_len = sample_batch['attention_mask'][i].sum().item()
    pred = predictions[i, :seq_len].cpu().numpy()
    true = sample_batch['label_masks'][i, :seq_len].cpu().numpy()

    print(f"Example {i+1}:")
    print(f"  Predicted: {pred[:50]}..." if len(pred) > 50 else f"  Predicted: {pred}")
    print(f"  True:      {true[:50]}..." if len(true) > 50 else f"  True:      {true}")
    print(f"  Accuracy:  {(pred == true).mean()*100:.2f}%")
    print()